# Multiple Linear Regression — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> ⚠️ **This file contains complete solutions. Release to students only after the submission deadline.**

In [ ]:
!pip install yfinance statsmodels pandas-datareader --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'

# Shared data — downloaded once
START = '2019-01-01'; END = '2024-12-31'
STOCK = 'NESN.SW'

stock_ret = (yf.download(STOCK, start=START, end=END, auto_adjust=True, progress=False)['Close']
             .squeeze().pct_change().dropna() * 100)
ff = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', START, END)[0]

data = pd.concat([stock_ret.rename('STOCK'), ff], axis=1).dropna()
data['EXCESS'] = data['STOCK'] - data['RF']
print(f'✓ Data loaded: {STOCK}, {len(data)} common trading days.')

---
# Solution 1 — Predict the Direction of Omitted-Variable Bias

Bias $= \beta_2 \cdot \mathrm{Cov}(x_1,x_2)/\mathrm{Var}(x_1)$ — the **sign is the product of the two signs** (Var is always positive).

| # | $\beta_2$ | Cov | Bias direction | Justification |
|---|-----------|-----|----------------|---------------|
| a | > 0 | > 0 | **Upward** | positive × positive — the market slope absorbs part of the momentum effect |
| b | < 0 | > 0 | **Downward** | negative × positive — the yield slope is pushed below its true value |
| c | > 0 | < 0 | **Downward** | positive × negative — omitting the factor drags the slope down |
| d | ≠ 0 | = 0 | **Unbiased** | zero covariance kills the bias even though the factor matters for y |
| e | > 0 | > 0 | **Upward** | pass-through coefficient overstated when funding costs are omitted |
| f | = 0 | > 0 | **Unbiased** | an irrelevant variable ($\beta_2 = 0$) cannot bias anything |

**Key insight:** BOTH conditions must hold. Omitting a relevant-but-uncorrelated variable costs fit ($R^2$), but does not bias the slope. Omitting an irrelevant-but-correlated variable is harmless.

---
# Solution 2 — FF3 Regression on a Swiss Stock

In [ ]:
y = data['EXCESS']
X_ff3 = sm.add_constant(data[['Mkt-RF', 'SMB', 'HML']])
model_ff3 = sm.OLS(y, X_ff3).fit()
print(model_ff3.summary())

**Answers:** (numbers are for Nestlé NESN.SW; other stocks differ)
1. The market loading is typically ≈ 0.35–0.45 and highly significant. SMB is negative (Nestlé is a mega-cap, so it moves *against* the small-firm factor) and usually significant; HML is small and often borderline.
2. Partial effect of SMB: *if the size factor rises by 1pp — a good day for small caps — while the market and value factors stay fixed, Nestlé's excess return changes by $\hat{\beta}_{SMB}$ pp on average.* For a negative loading: Nestlé underperforms on small-cap days, ceteris paribus.
3. With an HML loading close to zero (often slightly positive), Nestlé is roughly style-neutral — neither a pronounced value nor growth stock. (Contrast Apple's clearly negative, growth-type loading.)
4. $R^2$ is much lower (≈ 0.10–0.20 vs. 0.65 for Apple) for two reasons: the factors are **US-based** (imperfect proxy for Swiss market risk), and the **time-zone mismatch** — SIX closes before New York, so part of the US factor move is only reflected in the next Swiss trading day.

---
# Solution 3 — Omitted-Variable Bias Live

In [ ]:
X_capm = sm.add_constant(data['Mkt-RF'])
model_capm = sm.OLS(y, X_capm).fit()

b_slr = model_capm.params['Mkt-RF']
b_mlr = model_ff3.params['Mkt-RF']

print(f'CAPM (SLR):  β_Mkt_hat = {b_slr:.4f}')
print(f'FF3  (MLR):  β_Mkt_hat = {b_mlr:.4f}')
print(f'Bias = SLR − MLR = {b_slr - b_mlr:+.4f}')
print(f'\ncorr(Mkt-RF, SMB) = {data["Mkt-RF"].corr(data["SMB"]):+.3f}')
print(f'corr(Mkt-RF, HML) = {data["Mkt-RF"].corr(data["HML"]):+.3f}')
print(f'β_SMB_hat = {model_ff3.params["SMB"]:+.4f},  β_HML_hat = {model_ff3.params["HML"]:+.4f}')

**Answers:**
1. The market beta shifts by a few hundredths (direction depends on the stock — for Nestlé usually slightly downward when SMB/HML enter, because the negative SMB loading combined with the negative Mkt-SMB correlation was inflating the CAPM slope).
2. The sign of the total bias is the sum over omitted factors of $\hat{\beta}_j \times \mathrm{corr}(Mkt, x_j)$-terms. Read BOTH signs off the cell above — the loading from the FF3 table and the correlation from the correlation matrix — and multiply them: a positive product means the CAPM slope was biased upward, a negative product downward. Do not assume the sign of $\mathrm{corr}(Mkt, SMB)$; over daily post-2019 samples it is usually positive, not negative. HML contributes analogously with its own two signs.
3. Warning: *your CAPM beta silently absorbs every style exposure that correlates with the market — hedge with it and you are mis-hedged whenever size or value moves without the market.*

---
# Solution 4 — Adjusted $R^2$ by Hand

In [ ]:
TSS, RSS, k = 0.6200, 0.2480, 3

for n_ in [800, 30]:
    R2 = 1 - RSS / TSS
    penalty = (n_ - 1) / (n_ - k - 1)
    R2_adj = 1 - (1 - R2) * penalty
    print(f'n = {n_:>4}:  R² = 1 − {RSS}/{TSS} = {R2:.4f}')
    print(f'          penalty = {n_-1}/{n_-k-1} = {penalty:.4f}')
    print(f'          adj. R² = 1 − {1-R2:.4f} · {penalty:.4f} = {R2_adj:.4f}\n')

**Answers:**
1. For $n = 800$: $R^2 = 0.6000$ and $\bar{R}^2 = 0.5985$ — a gap of only 0.0015. With 796 residual degrees of freedom, three regressors cost almost nothing.
2. For $n = 30$: the penalty is $29/26 \approx 1.115$, so $\bar{R}^2 = 1 - 0.40 \times 1.115 = 0.5538$ — the gap widens to almost 5 percentage points. Degrees of freedom are scarce: each regressor consumes a meaningful share of the information in the sample.
3. Yes — $\bar{R}^2 < 0$ occurs when $R^2 < 1 - \frac{n-k-1}{n-1}$, i.e. when the model explains less than the df it burns. A negative $\bar{R}^2$ says the model is *worse than useless*: you would predict better with the sample mean alone.

---
# Solution 5 — The Model Ladder

In [ ]:
specs = {
    'CAPM (Mkt-RF)':     ['Mkt-RF'],
    'FF2 (Mkt+SMB)':     ['Mkt-RF', 'SMB'],
    'FF3 (Mkt+SMB+HML)': ['Mkt-RF', 'SMB', 'HML'],
}
rows = []
for name, cols in specs.items():
    m = sm.OLS(y, sm.add_constant(data[cols])).fit()
    rows.append({'Model': name, 'k': len(cols), 'R²': m.rsquared, 'adj. R²': m.rsquared_adj})
ladder = pd.DataFrame(rows).set_index('Model')
print(ladder.round(4))

fig, ax = plt.subplots(figsize=(9, 4.2))
xpos = np.arange(len(ladder)); w = 0.38
ax.bar(xpos - w/2, ladder['R²'],      w, color=ORANGE, label='$R^2$')
ax.bar(xpos + w/2, ladder['adj. R²'], w, color=YELLOW, edgecolor=GREY, lw=0.5, label='$\\bar{R}^2$')
ax.set_xticks(xpos); ax.set_xticklabels(ladder.index, fontsize=9)
ax.legend(loc='upper left', frameon=False)
ax.set_title(f'Model ladder — {STOCK}', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

for i in range(1, len(ladder)):
    d = ladder['adj. R²'].iloc[i] - ladder['adj. R²'].iloc[i-1]
    verdict = 'pays for itself ✓' if d > 0 else 'does NOT pay ✗'
    print(f'{ladder.index[i]}: Δ adj. R² = {d:+.4f} → {verdict}')

**Answers:**
1. For Nestlé, SMB typically pays for itself (clear adj.-$R^2$ gain); HML's contribution is small and can go either way depending on the window.
2. The ranking differs from Apple: Apple's dominant secondary exposure is the (negative) value loading, Nestlé's is the (negative) size loading. Factor loadings are firm characteristics, not universal constants.
3. $R^2$ is weakly increasing by construction: each larger model *nests* the smaller one, so OLS can always reproduce the smaller model's fit (set the extra coefficient to zero) and usually improves on it.

---
# Solution 6 — The Irrelevant-Regressor Experiment

In [ ]:
np.random.seed(1)
base_cols = ['Mkt-RF', 'SMB', 'HML']
m_base = sm.OLS(y, sm.add_constant(data[base_cols])).fit()

noise = np.random.normal(0, 1, len(data))
m_noise = sm.OLS(y, sm.add_constant(data[base_cols].assign(NOISE=noise))).fit()
print(f'FF3:          R² = {m_base.rsquared:.5f}   adj. R² = {m_base.rsquared_adj:.5f}')
print(f'FF3 + noise:  R² = {m_noise.rsquared:.5f}   adj. R² = {m_noise.rsquared_adj:.5f}')

# Repeat 100 times
deltas, tstats = [], []
for _ in range(100):
    nz = np.random.normal(0, 1, len(data))
    m  = sm.OLS(y, sm.add_constant(data[base_cols].assign(NOISE=nz))).fit()
    deltas.append(m.rsquared - m_base.rsquared)
    tstats.append(m.tvalues['NOISE'])
deltas, tstats = np.array(deltas), np.array(tstats)

print(f'\n100 noise draws:  ΔR² min = {deltas.min():.2e},  mean = {deltas.mean():.2e},  max = {deltas.max():.2e}')
print(f'Draws with ΔR² < 0: {(deltas < 0).sum()} of 100')
print(f'Draws with |t_NOISE| > 1.96: {(np.abs(tstats) > 1.96).sum()} of 100')

**Answers:**
1. **Zero** draws decrease $R^2$. Adding any regressor can never raise RSS — OLS can always set its coefficient to zero, and in-sample it almost never does exactly that. The increase is mechanical.
2. The mean fake gain is tiny (order $10^{-4}$–$10^{-3}$) — but always $\geq 0$. Over many candidate variables these fake gains accumulate, which is exactly why raw $R^2$ is unusable for model selection.
3. The t-statistic of NOISE is roughly standard normal across draws — so about **5 of 100** draws are 'significant' at the 5% level *by pure chance*. This is the **multiple-testing / data-mining problem**: search enough noise variables and some will always look significant.

---
# Solution 7 — F-Test by Hand

In [ ]:
R2_U = model_ff3.rsquared
R2_R = model_capm.rsquared
q    = 2
df_  = int(model_ff3.df_resid)

num = (R2_U - R2_R) / q
den = (1 - R2_U) / df_
F_obs  = num / den
F_crit = stats.f.ppf(0.95, q, df_)
p_val  = 1 - stats.f.cdf(F_obs, q, df_)

print('Step 1.  H0: β_SMB = β_HML = 0   vs.   H1: at least one ≠ 0')
print(f'Step 2.  α = 5% → F_crit({q}, {df_}) = {F_crit:.2f}')
print(f'Step 3.  numerator   = ({R2_U:.4f} − {R2_R:.4f})/2 = {num:.6f}')
print(f'         denominator = (1 − {R2_U:.4f})/{df_} = {den:.6f}')
print(f'         F_obs = {F_obs:.2f},   p = {p_val:.2e}')
decision = 'REJECT H0' if F_obs > F_crit else 'DO NOT REJECT H0'
print(f'Step 4.  F_obs {">" if F_obs > F_crit else "<"} F_crit → {decision}')

print('\nVerify with statsmodels:')
print(model_ff3.f_test('SMB = HML = 0'))

**Answers:**
1. For Nestlé, $F_{obs}$ is typically in the range 15–40 with $F_{crit} \approx 3.0$ → clearly **reject**: size and value jointly matter beyond the market, even for a defensive Swiss stock.
2. The numerator is the loss of explained variation *per imposed restriction* when SMB and HML are forced to zero; the denominator is the model's leftover noise *per residual degree of freedom*. F asks: is the lost fit large relative to noise?
3. Yes. With correlated regressors, each individual t-test asks 'does this variable add anything *given the other one is in the model*?' — and each can look redundant given its twin. The F-test asks whether the *pair* adds anything, which can be a resounding yes. (Exercise 8 constructs exactly this case.)

---
# Solution 8 — Joint vs. Individual Significance

In [ ]:
spy_ret = (yf.download('SPY', start=START, end=END, auto_adjust=True, progress=False)['Close']
           .squeeze().pct_change().dropna() * 100)
d8 = pd.concat([data[['EXCESS', 'Mkt-RF']], spy_ret.rename('SPY')], axis=1).dropna()

X8 = sm.add_constant(d8[['Mkt-RF', 'SPY']])
m8 = sm.OLS(d8['EXCESS'], X8).fit()

print(f'corr(Mkt-RF, SPY) = {d8["Mkt-RF"].corr(d8["SPY"]):.4f}   ← near-duplicates\n')
for f in ['Mkt-RF', 'SPY']:
    print(f'{f:<7}: β_hat = {m8.params[f]:+.4f},  t = {m8.tvalues[f]:+.2f},  p = {m8.pvalues[f]:.4f}')

print('\nJoint F-test H0: β_Mkt = β_SPY = 0:')
print(m8.f_test('Mkt-RF = 0, SPY = 0'))

**Answers:**
1. Individually, both slopes often have large p-values (sometimes both > 0.05) — yet the joint F-test p-value is essentially zero.
2. The two regressors are ~0.99 correlated: given SPY, Mkt-RF adds almost no *incremental* information (and vice versa), so each individual t-test says 'redundant'. But dropping *both* would discard the market signal entirely — the pair is indispensable, and the F-test sees that.
3. This is the classic multicollinearity symptom from the lecture: **high $R^2$ (strong joint significance) with weak individual t-statistics.**

---
# Solution 9 — VIF: By Hand and by Library

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_table(df_x):
    Xc = sm.add_constant(df_x)
    rows = []
    for j, col in enumerate(df_x.columns):
        others = [c for c in df_x.columns if c != col]
        r2j = sm.OLS(df_x[col], sm.add_constant(df_x[others])).fit().rsquared
        rows.append({'Regressor': col,
                     'R²_j': round(r2j, 4),
                     'VIF (hand)': round(1/(1-r2j), 2),
                     'VIF (statsmodels)': round(variance_inflation_factor(Xc.values, j+1), 2),
                     '√VIF (SE inflation)': round(np.sqrt(1/(1-r2j)), 2)})
    return pd.DataFrame(rows).set_index('Regressor')

print('Market-proxy pair (Exercise 8):')
print(vif_table(d8[['Mkt-RF', 'SPY']]))
print('\nFF3 factors:')
print(vif_table(data[['Mkt-RF', 'SMB', 'HML']]))

**Answers:**
1. The Mkt-RF/SPY pair has VIFs in the range of 40–90 (auxiliary $R^2 \approx 0.98$–0.99) — far beyond the danger threshold of 10. The FF3 factors sit near 1.1–1.3 — completely safe.
2. For a VIF of ~60, $\sqrt{VIF} \approx 7.7$: *the standard errors of the market-proxy slopes are almost eight times larger than they would be with orthogonal regressors* — which is exactly why the individual t-stats collapsed in Exercise 8.
3. Remedy for the proxy pair: **drop one of the two** (they measure the same thing) or combine them into a single market factor. For FF3, 'do nothing' is correct — the mild overlap barely inflates the SEs, and each factor carries distinct economic information.

---
# Solution 10 — Example: Do Gold Miners Load on Gold Beyond the Market?

In [ ]:
# Hypothesis: GDX (gold miners) loads positively on gold returns even after
# controlling for the broad equity market.
px = yf.download(['GDX', 'GLD', 'SPY'], start=START, end=END,
                 auto_adjust=True, progress=False)['Close']
ret10 = px.pct_change().dropna() * 100

y10 = ret10['GDX']
X10 = sm.add_constant(ret10[['SPY', 'GLD']])
m10 = sm.OLS(y10, X10).fit(cov_type='HC1')

print(m10.summary())
print('\nJoint F-test H0: β_SPY = β_GLD = 0 (HC1):')
print(m10.f_test('SPY = 0, GLD = 0'))

from statsmodels.stats.outliers_influence import variance_inflation_factor
Xc = sm.add_constant(ret10[['SPY', 'GLD']])
for j, c in enumerate(['SPY', 'GLD']):
    print(f'VIF({c}) = {variance_inflation_factor(Xc.values, j+1):.2f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(m10.fittedvalues, m10.resid, s=8, alpha=0.35, color=GREY)
ax.axhline(0, color=RED, lw=1.5)
ax.set_xlabel('Fitted Y_hat'); ax.set_ylabel('Residual u_hat')
ax.set_title('Residuals vs fitted — GDX two-factor model', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

**Executive summary:**
Gold miners load massively on gold — $\hat{\beta}_{GLD} \approx 1.6$–1.9 with a robust t-statistic far above 10 — while also carrying a market loading around 0.7: miners are leveraged gold plays wrapped in equity risk. Both slopes are individually and jointly significant (F-test p ≈ 0), and the two regressors are nearly uncorrelated (VIF ≈ 1), so the partial effects are cleanly identified. The model explains roughly half of GDX's daily variance; the rest is mining-specific risk (costs, production, jurisdictions). Caveat: loadings are unstable across monetary-policy regimes, so rolling re-estimation is advisable before using these betas for hedging.

---
# 🔥 Challenge Solution — FF5 vs. FF3

In [ ]:
ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', START, END)[0]

# Apple excess returns on the same sample as the 5-factor data
aapl = (yf.download('AAPL', start=START, end=END, auto_adjust=True, progress=False)['Close']
        .squeeze().pct_change().dropna() * 100)
d5 = pd.concat([aapl.rename('AAPL'), ff5], axis=1).dropna()
d5['EXCESS'] = d5['AAPL'] - d5['RF']

m3 = sm.OLS(d5['EXCESS'], sm.add_constant(d5[['Mkt-RF', 'SMB', 'HML']])).fit()
m5 = sm.OLS(d5['EXCESS'], sm.add_constant(d5[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])).fit()

print(f'FF3:  R² = {m3.rsquared:.4f}   adj. R² = {m3.rsquared_adj:.4f}')
print(f'FF5:  R² = {m5.rsquared:.4f}   adj. R² = {m5.rsquared_adj:.4f}')
print(f'\nβ_RMW_hat = {m5.params["RMW"]:+.4f}  (t = {m5.tvalues["RMW"]:.2f})')
print(f'β_CMA_hat = {m5.params["CMA"]:+.4f}  (t = {m5.tvalues["CMA"]:.2f})')

# F-test by hand
q, df_ = 2, int(m5.df_resid)
F_obs = ((m5.rsquared - m3.rsquared) / q) / ((1 - m5.rsquared) / df_)
F_crit = stats.f.ppf(0.95, q, df_)
print(f'\nF-test H0: β_RMW = β_CMA = 0')
print(f'F_obs = {F_obs:.2f}   vs.   F_crit = {F_crit:.2f}')
print('→ ' + ('REJECT H0 — profitability and investment add signal beyond FF3.'
       if F_obs > F_crit else 'DO NOT REJECT — RMW and CMA add nothing beyond FF3 here.'))
print('\nVerify with statsmodels:')
print(m5.f_test('RMW = 0, CMA = 0'))
print(f'\nΔ adj. R² (FF5 − FF3) = {m5.rsquared_adj - m3.rsquared_adj:+.4f}')

**How to read your output:** take the RMW and CMA loadings and the joint F-test from the cell above. Resist the tempting shortcut "Apple is highly profitable, so RMW must load positively": a time-series factor loading measures co-movement with the long-minus-short profitability portfolio, not the firm's own profitability, and for a mega-cap growth name the RMW loading is often small or negative. Whether the two extra factors add statistically significant signal is what the F-test decides; adjusted $R^2$ typically rises only slightly. The economic gain is modest (a percentage point or two of explained variance): the market factor still does the heavy lifting. This mirrors the literature — FF5 improves pricing mainly in the *cross-section* of many stocks, less in the time series of a single mega-cap.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*⚠️ Release to students only after the submission deadline.*